# Follow-up budget sweep on Colab

Choose **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all** and approve Drive access. T4 with FP16 matches `initialresultscolab.ipynb`. Use a fresh runtime if you have installed vLLM or changed PyTorch in the current session.

This runs the same four calibration problems (independent and dependent, lengths 2 and 4) through all three systems at **8,000 and 16,000 total tokens**. There are **24 new evaluations**. The experiment code is pinned to the commit recorded in your initial run, with the same model revision, prompts, thinking mode, and sampling seeds.

Each completed problem is saved to `MyDrive/small-multi-agents/runs/budget-sweep-8k-16k/`. Rerun the cells after a disconnect to resume. The final table also includes the existing 4k Drive logs when their environment matches. A100 is supported with BF16; a comparison on A100 should use `BUDGETS = [4000, 8000, 16000]` and a new run name so all budgets share the same GPU and precision.

In [ ]:
# Load the exact experiment code used by initialresultscolab.ipynb.
import os
from pathlib import Path
import subprocess
import sys

EXPERIMENT_REVISION = "6fab517568027ee12e89b7e0c5d5f0a3a93f4780"
REPO_DIR = Path("/content/small-multi-agents-budget-sweep")
if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/liatzheng/small-multi-agents.git", str(REPO_DIR)],
        check=True,
    )
subprocess.run(["git", "checkout", "--detach", EXPERIMENT_REVISION], cwd=REPO_DIR, check=True)
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
# Refresh solver imports when this notebook shares a previously used kernel.
for module_name in ("run_single_agent", "run_ensemble", "run_hierarchy"):
    sys.modules.pop(module_name, None)
requirements = [
    line.strip() for line in Path("requirements.txt").read_text().splitlines()
    if line.strip() and not line.startswith(("#", "torch=="))
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *requirements], check=True)
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
print("Experiment code", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())

In [ ]:
# Check the GPU, then connect persistent result storage.
import torch
from google.colab import drive

if not torch.cuda.is_available():
    raise RuntimeError("Choose a T4 or A100 GPU in Runtime → Change runtime type, then rerun.")
print("GPU", torch.cuda.get_device_name(0))
print("GPU memory (GiB)", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))
print("PyTorch", torch.__version__)
drive.mount("/content/drive")

In [ ]:
# Run settings. The two saved pairs are identical to the initial Colab run.
RUN_NAME = "budget-sweep-8k-16k"
BUDGETS = [8000, 16000]
SAMPLING_SEED = 42
BASELINE_DIR = Path("/content/drive/MyDrive/small-multi-agents/runs/calibration-4k")

import hashlib
import json
import tempfile
import transformers
import accelerate
from run_single_agent import MODEL, REVISION, read_problems

if not RUN_NAME or Path(RUN_NAME).name != RUN_NAME or RUN_NAME in (".", ".."):
    raise ValueError("Choose a simple folder name for RUN_NAME.")
if not BUDGETS or len(set(BUDGETS)) != len(BUDGETS) or any(type(b) is not int or not 1 <= b <= 32768 for b in BUDGETS):
    raise ValueError("BUDGETS must contain distinct integers from 1 to 32768.")
problems = read_problems(Path("results/chain_calibration/d2.jsonl"))
problems += read_problems(Path("results/chain_calibration/d4.jsonl"))
problem_text = "".join(json.dumps(problem) + "\n" for problem in problems)
run_dir = Path("/content/drive/MyDrive/small-multi-agents/runs") / RUN_NAME
run_dir.mkdir(parents=True, exist_ok=True)
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported(including_emulation=False) else torch.float16
run_config = {
    "model": MODEL, "revision": REVISION,
    "budgets": BUDGETS, "sampling_seed": SAMPLING_SEED,
    "problems_sha256": hashlib.sha256(problem_text.encode()).hexdigest(),
    "gpu": torch.cuda.get_device_name(0), "dtype": str(dtype),
    "torch": torch.__version__, "cuda": torch.version.cuda,
    "transformers": transformers.__version__, "accelerate": accelerate.__version__,
    "repo_commit": subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip(),
}

def save_text(path, text):
    with tempfile.NamedTemporaryFile(mode="w", encoding="utf-8", dir=path.parent, delete=False) as stream:
        temporary_path = Path(stream.name)
        stream.write(text)
        stream.flush()
        os.fsync(stream.fileno())
    temporary_path.replace(path)

config_path = run_dir / "config.json"
if config_path.exists() and json.loads(config_path.read_text()) != run_config:
    raise ValueError("This run has different settings or hardware. Choose a new RUN_NAME.")
save_text(config_path, json.dumps(run_config, indent=2) + "\n")
save_text(run_dir / "problems.jsonl", problem_text)
print(f"{len(problems)} problems × 3 systems × {len(BUDGETS)} budgets = {len(problems) * 3 * len(BUDGETS)} evaluations")
print("Precision", dtype)
print("Results", run_dir)

In [ ]:
# Load Qwen once for all systems. The first load downloads about 8 GB.
import gc
from run_single_agent import load_model, solve
from run_ensemble import solve_ensemble
from run_hierarchy import solve_hierarchy

if "model" in globals():
    del model
    gc.collect()
    torch.cuda.empty_cache()
model, tokenizer = load_model()
print("Loaded", MODEL, "on", model.device, "with", model.dtype)

In [ ]:
# Run each budget with all three systems, saving after every problem.
from time import perf_counter

solvers = {"single_agent": solve, "ensemble": solve_ensemble, "hierarchy": solve_hierarchy}
records = []
for budget in BUDGETS:
    budget_dir = run_dir / f"b{budget}"
    budget_dir.mkdir(exist_ok=True)
    for system, solver in solvers.items():
        system_records = []
        for problem in problems:
            label = f"{system}_d{problem['chain_length']}_{problem['pair_id']}_{problem['kind']}"
            checkpoint = budget_dir / f"{label}.json"
            expected = {
                **problem, "system": system, "budget": budget,
                "model": MODEL, "revision": REVISION, "dtype": str(model.dtype), "seed": SAMPLING_SEED,
            }
            if checkpoint.exists():
                record = json.loads(checkpoint.read_text())
                if any(record.get(key) != value for key, value in expected.items()):
                    raise ValueError(f"Checkpoint differs from this run: {checkpoint}")
                print("Resumed", budget, label, record["status"], flush=True)
            else:
                print("Running", budget, label, flush=True)
                started = perf_counter()
                record = solver(problem, model, tokenizer, budget, SAMPLING_SEED)
                record.update(
                    system=system, model=MODEL, revision=REVISION, seed=SAMPLING_SEED,
                    seconds=round(perf_counter() - started, 3),
                )
                if record["total_tokens"] > budget:
                    raise RuntimeError(f"Token budget exceeded for {label}")
                save_text(checkpoint, json.dumps(record, indent=2) + "\n")
                print(f"Saved {record['status']}, {record['total_tokens']} tokens", flush=True)
            records.append(record)
            system_records.append(record)
            save_text(
                budget_dir / f"{system}.jsonl",
                "".join(json.dumps(result) + "\n" for result in system_records),
            )
print("All evaluations complete.")

In [ ]:
# Include the earlier 4k results when the inputs and environment match.
comparison_records = list(records)
baseline_config_path = BASELINE_DIR / "config.json"
if 4000 in BUDGETS:
    print("Using this run's 4k results.")
elif not baseline_config_path.exists():
    print("The earlier 4k Drive folder was not found. The table will show the new budgets.")
else:
    baseline_config = json.loads(baseline_config_path.read_text())
    shared_keys = [key for key in run_config if key != "budgets"]
    differences = [key for key in shared_keys if baseline_config.get(key) != run_config[key]]
    baseline_logs = [BASELINE_DIR / f"{system}.jsonl" for system in solvers]
    if differences or any(not path.exists() for path in baseline_logs):
        print("The 4k logs are incomplete or use a different environment.", differences)
        print("The comparison table will show the new budgets.")
    else:
        baseline_records = []
        for system, path in zip(solvers, baseline_logs):
            previous = [json.loads(line) for line in path.read_text().splitlines() if line.strip()]
            if len(previous) != len(problems):
                raise ValueError(f"Expected four baseline records in {path}.")
            for problem, record in zip(problems, previous):
                expected = {
                    **problem, "system": system, "budget": 4000,
                    "model": MODEL, "revision": REVISION, "seed": SAMPLING_SEED,
                    "dtype": run_config["dtype"],
                }
                if any(record.get(key) != value for key, value in expected.items()):
                    raise ValueError(f"Baseline record differs from this experiment: {path}")
                baseline_records.append(record)
        comparison_records += baseline_records
        print("Included all 12 earlier 4k results.")


In [ ]:
# Accuracy, token usage, and the completion status of every attempt or stage.
import pandas as pd
from IPython.display import display

rows = []
for record in comparison_records:
    row = {key: record.get(key) for key in (
        "budget", "chain_length", "pair_id", "kind", "system", "correct", "status",
        "input_tokens", "output_tokens", "total_tokens", "seconds", "failed_stage",
    )}
    if record["system"] == "ensemble":
        attempts = record["attempts"]
        row["completed_attempts"] = sum(attempt["status"] != "length_limit" for attempt in attempts)
        row["valid_votes"] = sum(attempt["vote"] is not None for attempt in attempts)
        row["parts"] = ", ".join(f"attempt {i + 1}={attempt['status']}" for i, attempt in enumerate(attempts))
    elif record["system"] == "hierarchy":
        stages = {name: "not_run" for name in ("planner", "worker_1", "worker_2", "merge")}
        stages.update({call["stage"]: call["stop_reason"] for call in record["calls"]})
        row["parts"] = ", ".join(f"{name}={status}" for name, status in stages.items())
    else:
        row["parts"] = record["status"]
    rows.append(row)

frame = pd.DataFrame(rows).sort_values(["budget", "system", "chain_length", "kind"])
summary = frame.groupby(["budget", "system"]).agg(
    correct=("correct", "sum"), problems=("correct", "size"), accuracy=("correct", "mean"),
    mean_input_tokens=("input_tokens", "mean"), mean_output_tokens=("output_tokens", "mean"),
    mean_total_tokens=("total_tokens", "mean"), total_seconds=("seconds", "sum"),
).reset_index()
by_kind = frame.groupby(["budget", "chain_length", "kind", "system"]).agg(
    correct=("correct", "sum"), problems=("correct", "size"), accuracy=("correct", "mean"),
    mean_total_tokens=("total_tokens", "mean"),
).reset_index()
save_text(run_dir / "summary_problems.csv", frame.to_csv(index=False))
save_text(run_dir / "summary_by_system.csv", summary.to_csv(index=False))
save_text(run_dir / "summary_by_kind.csv", by_kind.to_csv(index=False))
pd.set_option("display.max_colwidth", None)
display(summary.round(3))
display(frame[["budget", "chain_length", "kind", "system", "status", "total_tokens", "failed_stage", "parts"]])
print("Saved result logs and comparison tables to", run_dir)

The table reports an equal total token cap for every system at each budget. `parts` shows ensemble attempt outcomes and hierarchy stage stop reasons. `failed_stage` identifies the hierarchy stage that ended the problem, including a plan-format failure after a completed model call. `not_run` means execution ended before that stage.

The original 4k logs stay in their original Drive folder. New logs and checkpoints are stored by budget under this run's folder. Use **Runtime → Disconnect and delete runtime** when finished to release the GPU. Save a copy of the notebook with outputs to GitHub when you want to share the comparison.